<a href="https://colab.research.google.com/github/Zakirza/LLM_FOR_PROBLEM_SOLVING/blob/main/Qwen2_5_LeetCode_Finetune_Unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Fine-tune Qwen2.5-Coder with Unsloth for LeetCode Problem Solving

This notebook fine-tunes `Qwen2.5-Coder-7B-Instruct` using **Unsloth** (2x faster, 60% less VRAM) on a LeetCode dataset.

## 📋 Table of Contents
1. Environment Setup
2. Load Model with Unsloth
3. Apply QLoRA
4. Prepare Dataset
5. Train
6. Save & Export
7. Inference / Test

---
**Recommended Runtime:** Google Colab A100 or Kaggle (2x T4)  
**Estimated Training Time:** ~1-2 hrs on A100 | ~3-4 hrs on T4

## Step 1 —  Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi
!nvcc --version

Tue Mar  3 19:52:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Install Unsloth (auto-detects CUDA version)
%%capture
!pip install unsloth
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Supporting libraries
!pip install datasets trl peft bitsandbytes accelerate

print("✅ Installation complete!")

In [ ]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
print(f"BF16 supported  : {torch.cuda.is_bf16_supported()}")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB
BF16 supported  : True


## Step 2 —  Load Qwen2.5-Coder with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── Configuration ──────────────────────────────────────────────────
MODEL_NAME     = "unsloth/Qwen2.5-Coder-7B-Instruct"  # Unsloth-optimized
MAX_SEQ_LENGTH = 2048   # LeetCode solutions can be lengthy
LOAD_IN_4BIT   = True   # QLoRA — saves VRAM significantly
# ───────────────────────────────────────────────────────────────────

# Available model variants:
# "unsloth/Qwen2.5-Coder-1.5B-Instruct"  → ~4GB VRAM  (fast testing)
# "unsloth/Qwen2.5-Coder-7B-Instruct"    → ~10GB VRAM (recommended)
# "unsloth/Qwen2.5-Coder-14B-Instruct"   → ~20GB VRAM (high accuracy)
# "unsloth/Qwen2.5-Coder-32B-Instruct"   → ~40GB VRAM (best)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,          # Auto-detect: bf16 on Ampere+, fp16 otherwise
    load_in_4bit    = LOAD_IN_4BIT,
)

print(f" Model loaded: {MODEL_NAME}")
print(f"   Parameters  : {model.num_parameters() / 1e9:.1f}B")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

✅ Model loaded: unsloth/Qwen2.5-Coder-7B-Instruct
   Parameters  : 7.6B


## Step 3 —  Apply QLoRA Adapters

In [ ]:
# ── LoRA Configuration ─────────────────────────────────────────────
LORA_RANK    = 16   # 8=light, 16=balanced, 32=heavy (more capacity)
LORA_ALPHA   = 32   # Typically 2x rank
LORA_DROPOUT = 0    # Unsloth recommends 0 for speed
# ───────────────────────────────────────────────────────────────────

model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_RANK,
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = LORA_DROPOUT,
    bias            = "none",
    target_modules  = [
        # Attention layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        # MLP layers — important for coding/reasoning tasks!
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing = "unsloth",  # Unsloth's optimized checkpointing
    random_state    = 42,
    use_rslora      = False,   # Set True to use Rank-Stabilized LoRA
    loftq_config    = None,
)

# Show trainable parameters
model.print_trainable_parameters()

Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## Step 4 —  Prepare LeetCode Dataset

In [ ]:
from datasets import load_dataset

# Load the LeetCode dataset from HuggingFace
# Source: https://huggingface.co/datasets/greengerong/leetcode
# Contains ~2000 problems with Python, Java, C++ solutions
raw_dataset = load_dataset("greengerong/leetcode", split="train")

print(f"Total samples  : {len(raw_dataset)}")
print(f"Columns        : {raw_dataset.column_names}")
print("\nSample entry:")
print(f"  Title        : {raw_dataset[0]['title']}")
print(f"  Content (100): {raw_dataset[0]['content'][:200]}...")

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

leetcode-train.jsonl:   0%|          | 0.00/16.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2360 [00:00<?, ? examples/s]

Total samples  : 2360
Columns        : ['id', 'slug', 'title', 'difficulty', 'content', 'java', 'c++', 'python', 'javascript']

Sample entry:
  Title        : Two Sum
  Content (100): Given an array of integers `nums` and an integer `target`, return _indices of the two numbers such that they add up to `target`_.

You may assume that each input would have **_exactly_ one solution**,...


In [ ]:
# Filter out problems without Python solutions
dataset = raw_dataset.filter(lambda x: x['python'] is not None and len(x['python']) > 50)
print(f"After filter   : {len(dataset)} samples with Python solutions")

# Optional: Use a subset for quick testing
# dataset = dataset.select(range(200))  # Uncomment to test on 200 samples

Filter:   0%|          | 0/2360 [00:00<?, ? examples/s]

After filter   : 2359 samples with Python solutions


In [ ]:
# System prompt — the "personality" of our LeetCode solver
SYSTEM_PROMPT = """You are an expert competitive programmer specializing in LeetCode.
For each problem you receive:
1. Identify the core algorithm pattern (DP, sliding window, two pointers, graph, etc.)
2. Briefly explain your approach and why it is optimal
3. Write clean, well-commented Python code
4. State the Time and Space complexity with justification
Always aim for the most efficient solution possible."""


def format_prompt(example):
    """Format each problem into Qwen's chat template."""
    problem_text = example['content'] if example['content'] else example['title']
    solution_text = example['python']

    text = f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
## LeetCode Problem: {example['title']}

{problem_text}
<|im_end|>
<|im_start|>assistant
{solution_text}<|im_end|>"""

    return {"text": text}


# Apply formatting
dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)

# Preview formatted sample
print("\n" + "="*60)
print("FORMATTED SAMPLE:")
print("="*60)
print(dataset[0]['text'][:800])
print("...")

Map:   0%|          | 0/2359 [00:00<?, ? examples/s]


FORMATTED SAMPLE:
<|im_start|>system
You are an expert competitive programmer specializing in LeetCode.
For each problem you receive:
1. Identify the core algorithm pattern (DP, sliding window, two pointers, graph, etc.)
2. Briefly explain your approach and why it is optimal
3. Write clean, well-commented Python code
4. State the Time and Space complexity with justification
Always aim for the most efficient solution possible.<|im_end|>
<|im_start|>user
## LeetCode Problem: Two Sum

Given an array of integers `nums` and an integer `target`, return _indices of the two numbers such that they add up to `target`_.

You may assume that each input would have **_exactly_ one solution**, and you may not use the _same_ element twice.

You can return the answer in any order.

**Example 1:**

**Input:** nums = \[2,7,11
...


In [ ]:
# Train / Eval split (90% / 10%)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset  = dataset['test']

print(f"Train samples  : {len(train_dataset)}")
print(f"Eval  samples  : {len(eval_dataset)}")

Train samples  : 2123
Eval  samples  : 236


## Step 5 — 🏋️ Train with SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

# ── Training Configuration ─────────────────────────────────────────
OUTPUT_DIR              = "./qwen25-coder-leetcode"
NUM_TRAIN_EPOCHS        = 3      # Increase to 5 for better results
PER_DEVICE_BATCH_SIZE   = 2      # Adjust based on GPU VRAM
GRADIENT_ACCUM_STEPS    = 4      # Effective batch = 2 * 4 = 8
LEARNING_RATE           = 2e-4
WARMUP_STEPS            = 10
SAVE_STEPS              = 100
LOGGING_STEPS           = 10
# ───────────────────────────────────────────────────────────────────

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 2,
        packing                     = True,   # Packs short sequences → faster training

        # Output
        output_dir                  = OUTPUT_DIR,

        # Training schedule
        num_train_epochs            = NUM_TRAIN_EPOCHS,
        per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size  = PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUM_STEPS,
        warmup_steps                = WARMUP_STEPS,

        # Optimizer
        learning_rate               = LEARNING_RATE,
        optim                       = "adamw_8bit",  # Unsloth 8-bit Adam
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",

        # Precision
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),

        # Logging & Saving
        logging_steps               = LOGGING_STEPS,
        save_steps                  = SAVE_STEPS,
        eval_strategy               = "steps",
        eval_steps                  = SAVE_STEPS,
        save_total_limit            = 3,
        load_best_model_at_end      = True,

        # Reproducibility
        seed                        = 42,
        report_to                   = "none",  # Set "wandb" to enable W&B logging
    ),
)

print("\n🏋️ Starting training...")
print(f"   Epochs           : {NUM_TRAIN_EPOCHS}")
print(f"   Effective batch  : {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUM_STEPS}")
print(f"   Learning rate    : {LEARNING_RATE}")
print(f"   Output dir       : {OUTPUT_DIR}\n")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2123 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/2123 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/236 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=2):   0%|          | 0/236 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!

🏋️ Starting training...
   Epochs           : 3
   Effective batch  : 8
   Learning rate    : 0.0002
   Output dir       : ./qwen25-coder-leetcode



In [ ]:
# Show GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} | VRAM: {max_memory} GB | Reserved: {start_gpu_memory} GB")

# 🚀 Run training!
trainer_stats = trainer.train()

# Show training results
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"\n✅ Training complete!")
print(f"   Peak VRAM used   : {used_memory} GB")
print(f"   VRAM for LoRA    : {used_memory_for_lora} GB")
print(f"   Training time    : {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


GPU: Tesla T4 | VRAM: 14.563 GB | Reserved: 5.375 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 883 | Num Epochs = 3 | Total steps = 333
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss,Validation Loss
100,0.283829,0.281987
200,0.255719,0.281521


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
100,0.283829,0.281987
200,0.255719,0.281521


## Step 6 —  Save & Export Model

In [ ]:
# ── Option A: Save LoRA adapters only (~100MB) ─────────────────────
# Best for: sharing, loading on top of base model later
model.save_pretrained("qwen25-leetcode-lora")
tokenizer.save_pretrained("qwen25-leetcode-lora")
print("✅ Saved: LoRA adapters → qwen25-leetcode-lora/")

In [ ]:
# ── Option B: Save merged model (full weights) ────────────────────
# Best for: deploying standalone, no need for base model
model.save_pretrained_merged(
    "qwen25-leetcode-merged",
    tokenizer,
    save_method = "merged_16bit",  # Full precision merge
)
print("✅ Saved: Merged model → qwen25-leetcode-merged/")

In [ ]:
# ── Option C: Export to GGUF (run locally with Ollama / llama.cpp) ─
# Quantization options: q2_k, q3_k_m, q4_k_m, q5_k_m, q8_0, f16
# q4_k_m = best balance of quality vs size
model.save_pretrained_gguf(
    "qwen25-leetcode-gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("✅ Saved: GGUF → qwen25-leetcode-gguf/")
print("   → Load in Ollama: ollama create leetcode-solver -f Modelfile")

In [ ]:
# ── Option D: Push to HuggingFace Hub ─────────────────────────────
# Uncomment and fill in your HF token + username

# HF_TOKEN    = "hf_your_token_here"   # Get from https://huggingface.co/settings/tokens
# HF_USERNAME = "your-username"
# REPO_NAME   = "qwen25-coder-leetcode-lora"

# model.push_to_hub(
#     f"{HF_USERNAME}/{REPO_NAME}",
#     token = HF_TOKEN,
# )
# tokenizer.push_to_hub(
#     f"{HF_USERNAME}/{REPO_NAME}",
#     token = HF_TOKEN,
# )
# print(f"✅ Pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

print("ℹ️  Uncomment the above code and fill in your HF credentials to push to Hub.")

## Step 7 —  Inference & Testing

In [ ]:
# Switch model to fast inference mode (Unsloth optimization)
FastLanguageModel.for_inference(model)
print("✅ Model set to inference mode (2x faster)")

In [ ]:
def solve_leetcode(problem_title: str, problem_description: str,
                   max_new_tokens: int = 768, temperature: float = 0.2):
    """Generate a solution for a LeetCode problem."""
    prompt = f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
## LeetCode Problem: {problem_title}

{problem_description}
<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens      = max_new_tokens,
            temperature         = temperature,    # Low = more deterministic code
            do_sample           = True,
            repetition_penalty  = 1.1,            # Reduce repetition
            pad_token_id        = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


print("✅ solve_leetcode() helper ready")

In [ ]:
# ── Test 1: Two Sum (Easy) ─────────────────────────────────────────
problem = """Given an array of integers `nums` and an integer `target`, return indices
of the two numbers such that they add up to target.

You may assume that each input would have exactly one solution, and you may not use the
same element twice. You can return the answer in any order.

Example:
  Input: nums = [2,7,11,15], target = 9
  Output: [0,1]

Constraints:
  2 <= nums.length <= 10^4
  -10^9 <= nums[i] <= 10^9
  Only one valid answer exists."""

print("=" * 60)
print("PROBLEM: Two Sum (Easy)")
print("=" * 60)
solution = solve_leetcode("Two Sum", problem)
print(solution)

In [ ]:
# ── Test 2: Longest Substring Without Repeating (Medium) ──────────
problem = """Given a string s, find the length of the longest substring without
repeating characters.

Example 1:
  Input: s = "abcabcbb"
  Output: 3  ("abc")

Example 2:
  Input: s = "bbbbb"
  Output: 1  ("b")

Constraints:
  0 <= s.length <= 5 * 10^4
  s consists of English letters, digits, symbols and spaces."""

print("=" * 60)
print("PROBLEM: Longest Substring Without Repeating Characters (Medium)")
print("=" * 60)
solution = solve_leetcode("Longest Substring Without Repeating Characters", problem)
print(solution)

In [ ]:
# ── Test 3: Coin Change (Medium — DP) ────────────────────────────
problem = """You are given an integer array coins representing coins of different
denominations and an integer amount representing a total amount of money.

Return the fewest number of coins that you need to make up that amount.
If that amount of money cannot be made up by any combination of the coins, return -1.

Example:
  Input: coins = [1,5,11], amount = 15
  Output: 3  (5 + 5 + 5)

Constraints:
  1 <= coins.length <= 12
  1 <= coins[i] <= 2^31 - 1
  0 <= amount <= 10^4"""

print("=" * 60)
print("PROBLEM: Coin Change (Medium — Dynamic Programming)")
print("=" * 60)
solution = solve_leetcode("Coin Change", problem)
print(solution)

## Bonus —  Training Metrics Visualization

In [ ]:
import matplotlib.pyplot as plt

# Extract training logs
logs = trainer.state.log_history

train_loss  = [(l['step'], l['loss'])      for l in logs if 'loss'      in l and 'eval_loss' not in l]
eval_loss   = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Qwen2.5-Coder LeetCode Fine-tuning', fontsize=14, fontweight='bold')

# Training loss
if train_loss:
    steps, losses = zip(*train_loss)
    axes[0].plot(steps, losses, color='#4A90D9', linewidth=2, label='Train Loss')
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Steps')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

# Eval loss
if eval_loss:
    steps, losses = zip(*eval_loss)
    axes[1].plot(steps, losses, color='#E94F37', linewidth=2, label='Eval Loss')
    axes[1].set_title('Validation Loss')
    axes[1].set_xlabel('Steps')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: training_metrics.png")

## 📋 Summary

| Step | What we did |
|------|-------------|
| 1 | Installed Unsloth + dependencies |
| 2 | Loaded `Qwen2.5-Coder-7B-Instruct` in 4-bit |
| 3 | Applied QLoRA (rank=16) to attention + MLP layers |
| 4 | Formatted LeetCode dataset with Qwen chat template |
| 5 | Trained with SFTTrainer (3 epochs, cosine LR, 8-bit Adam) |
| 6 | Saved as LoRA adapters / merged model / GGUF |
| 7 | Tested on Easy, Medium, and DP problems |

### 🔗 Next Steps
- **Increase dataset quality**: Add GPT-4 generated explanations + complexity analysis
- **Add more epochs**: Try 5 epochs with a lower LR for better convergence  
- **Evaluate with Pass@k**: Test correctness by submitting to LeetCode programmatically
- **Deploy**: Use GGUF with [Ollama](https://ollama.com) for local inference
- **Share**: Push to HuggingFace Hub so others can use your model